<a href="https://colab.research.google.com/github/kinchittrivedi/Kaggle/blob/main/Feature_Extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
jeffpoulshaju_cleaning_text_data_path = kagglehub.notebook_output_download('jeffpoulshaju/cleaning-text-data')

print('Data source import complete.')


Extracting files...
Data source import complete.


# Feature Extraction

Welcome to the Feature Extraction phase! At this point, we have successfully cleaned and preprocessed our text messages (lowercasing, removing punctuation, filtering stopwords, and stemming). However, we still have one major problem: **machine learning models cannot read text.**

Models are built on mathematics, meaning they require numerical input to function. Feature extraction (also known as vectorization) is the process of translating our cleaned text into numbers.

In this notebook, we will explore two of the most foundational techniques in Natural Language Processing to accomplish this:
1. **Bag of Words (CountVectorizer):** Counting the frequency of words.
2. **TF-IDF (Term Frequency-Inverse Document Frequency):** Weighing the importance of words.

Let's dive in and turn our words into data!

In [4]:
import pandas as pd
import os

# The internal Kaggle path to the CSV in our previous notebook
# file_path = '/kaggle/input/notebooks/jeffpoulshaju/cleaning-text-data/cleaned_spam_data.csv'
file_path = os.path.join(jeffpoulshaju_cleaning_text_data_path, 'cleaned_spam_data.csv')

# Load our pre-cleaned dataset
df = pd.read_csv(file_path)

# PRO-TIP: Handling blanks
# If a spam message was completely wiped out by the stopword remover,
# Pandas reads the blank cell as 'NaN'. We fix this by replacing NaNs with an empty string:
df['processed_message'] = df['processed_message'].fillna('')

# Inspect the loaded data
df.head()

,label,message,tokens,processed_message
0,ham,go until jurong point crazy available only in ...,"['go', 'jurong', 'point', 'crazi', 'avail', 'b...",go jurong point crazi avail bugi n great world...
1,ham,ok lar joking wif u oni,"['ok', 'lar', 'joke', 'wif', 'u', 'oni']",ok lar joke wif u oni
2,spam,free entry in 2 a wkly comp to win fa cup fina...,"['free', 'entri', '2', 'wkli', 'comp', 'win', ...",free entri 2 wkli comp win fa cup final tkt 21...
3,ham,u dun say so early hor u c already then say,"['u', 'dun', 'say', 'earli', 'hor', 'u', 'c', ...",u dun say earli hor u c alreadi say
4,ham,nah i dont think he goes to usf he lives aroun...,"['nah', 'dont', 'think', 'goe', 'usf', 'live',...",nah dont think goe usf live around though


## Bag of Words (BoW)

The **Bag of Words (BoW)** model is one of the simplest ways to convert text into numbers. Imagine taking a sentence, chopping up all the words, throwing them into a bag, and simply counting how many times each word appears.

It works in two straightforward steps:
1. **Builds a Vocabulary:** It scans our entire dataset and creates a master list of every unique word used across all messages.
2. **Counts Frequencies:** For each individual message, it creates a numerical array (a tally) showing how many times those vocabulary words appear.

*For example, if our master vocabulary is `['free', 'money', 'today', 'win']`:*
* *The message `"win free money today"` becomes the array `[1, 1, 1, 1]`.*
* *The message `"free money free"` becomes the array `[2, 1, 0, 0]`.*

Notice that this approach completely ignores grammar and word order—it only cares about *what* words are present and *how often* they occur. In scikit-learn, we build this using the `CountVectorizer` tool.

In [5]:
# Import the CountVectorizer tool from scikit-learn to build our Bag of Words
from sklearn.feature_extraction.text import CountVectorizer

# Initialize the vectorizer object
cv = CountVectorizer()

# Fit the vectorizer to our cleaned text and transform it into a numerical matrix
X_bow = cv.fit_transform(df['processed_message'])

# Print the shape of the matrix to see the number of text messages and unique words
print("Matrix Shape:")
print(X_bow.shape)

# Display the first 20 unique words the vectorizer learned for its master vocabulary
print("\nFirst 20 Vocabulary Words:")
print(cv.get_feature_names_out()[:20])

Matrix Shape:
(5169, 8065)

First 20 Vocabulary Words:
['008704050406' '0089mi' '0121' '01223585236' '01223585334' '0125698789'
 '02' '020603' '0207' '02070836089' '02072069400' '02073162414'
 '02085076972' '020903' '021' '050703' '0578' '06' '060505' '061104']


## The Document Term Matrix (DTM)

It is important to understand that we are not creating any brand new information here. We are simply taking the exact same text messages we already have and **rearranging them into a giant numerical grid so the computer can understand them.**



In the machine learning world, this grid is called a **Document Term Matrix**, or DTM for short. You can think of it exactly like a massive Excel spreadsheet. Every single row in this spreadsheet is just one of our text messages. Every single column is simply a word from our master vocabulary. The numbers inside the boxes just tell us how many times a particular word showed up in that particular message.

Let us run the code to peak inside our very own Document Term Matrix and see how our text has been rearranged into raw numbers.

In [ ]:
print(X_bow.toarray())

## TF-IDF (Term Frequency-Inverse Document Frequency)

While Bag of Words is a great starting point, it has **one major flaw: it treats all words equally.** If a word appears a hundred times, it gets a massive score, even if it is a common word that does not actually help us identify spam.

**TF-IDF** fixes this by scoring words based on how *meaningful* they are, rather than just how often they show up. It balances two distinct metrics:

1. **Term Frequency (TF):** How often does a term ($t$) appear in this *specific* document ($d$)? (More frequent = higher score).
   $$TF(t, d)=\frac{\text{Number of times term } t \text{ appears in document } d}{\text{Total number of terms in document } d}$$

2. **Inverse Document Frequency (IDF):** How often does this term ($t$) appear across the *entire* dataset of $N$ documents? (More frequent everywhere = lower score).
   $$IDF(t)=\log\left(\frac{\text{Total number of documents } (N)}{\text{Number of documents containing term } t}\right)$$

By multiplying these together, we get our final mathematical score for each word:

$$TF\text{-}IDF(t, d)=TF(t, d)\times IDF(t)$$

**How it works in practice:**
Imagine the word "winner". If it appears three times in a single text message, its **TF** is high. If it rarely appears in the rest of our dataset, its **IDF** remains high. TF-IDF multiplies these together to give "winner" a very strong weight.

Conversely, if a word appears frequently in one message but is also found in almost every other message in our dataset, the IDF penalty pushes its final score down, telling our model, *"Ignore this word; it is too common to be useful."*

In [ ]:
# Import the TF-IDF Vectorizer tool from the scikit-learn library
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize the vectorizer and limit our vocabulary to the top 3000 most important words
tfidf = TfidfVectorizer(max_features=3000)

# Fit the vectorizer to our cleaned text and transform it into a weighted numerical matrix
X = tfidf.fit_transform(df['processed_message'])

# Print the shape to verify our rows of text messages and our 3000 vocabulary words
print("TF-IDF Matrix Shape:")
print(X.shape)

# Display a sample of the first 20 unique words the vectorizer chose for the vocabulary
print("\nFirst 20 TF-IDF Vocabulary Words:")
print(tfidf.get_feature_names_out()[:20])

## Encoding the Target Label

At this point, we have successfully converted all of our text messages (our `X` features) into numbers using TF-IDF. However, if we look at our target variable (our `y` labels), they are still stored as the text strings `"ham"` and `"spam"`.

Just like our features, **our final answers must also be numerical before we can feed them into a machine learning model.**

To fix this, we will use scikit-learn's `LabelEncoder`. This tool simply maps our text categories to integers. In binary classification like ours, it is standard practice to map the normal/negative class (`"ham"`) to **0**, and the target/positive class (`"spam"`) to **1**.

Once this quick step is done, our entire dataset will be 100% numerical and fully ready for the modeling phase!

In [ ]:
# Import the LabelEncoder tool from scikit-learn
from sklearn.preprocessing import LabelEncoder

# Initialize the encoder object
encoder = LabelEncoder()

# Learn the unique text labels in our dataset and convert them into numbers
# This transforms our "ham" and "spam" labels into 0s and 1s and saves them as y
y = encoder.fit_transform(df['label'])

## Saving Our Data

With our text successfully converted into numerical features and our labels encoded into numbers, we have everything we need to train our model. We will now save these mathematical arrays so they are ready for the next step of our Learn Guide. We use standard NumPy for our simple label array and a special SciPy command to compress our massive feature matrix efficiently.

In [ ]:
import numpy as np
import scipy.sparse

# 1. Save the encoded labels (y)
np.save('y_encoded.npy', y)

# 2. Save the TF-IDF features (X)
# Note: Because X is a massive, sparse mathematical matrix, we use SciPy to save it efficiently.
scipy.sparse.save_npz('X_features.npz', X)

print("Success! Features and labels saved for the modeling stage.")

## What's Next?

Our text has officially been transformed into numbers! Now that we have our mathematical features ready, it's time to build and train our spam filter using a powerful algorithm.

Ready? Let's jump into the next step:[Modeling with XGBoost](https://www.kaggle.com/code/jeffpoulshaju/modeling-with-xgboost)